<a href="https://colab.research.google.com/github/SampurnaKumar-2007/Automated-sales-analytics/blob/main/automated_sales_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install openpyxl so Python can save Excel files
!pip install openpyxl

# Import pandas into our code
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Set a random seed so the numbers are predictable
np.random.seed(42)

# Generate synthetic raw data (500 orders)
n_rows = 500
dates = pd.date_range(start="2025-01-01", periods=180, freq="D")

raw_data = {
    "Order_ID": [f"ORD-{1000 + i}" for i in range(n_rows)],
    "Date": np.random.choice(dates, size=n_rows),
    "Region": np.random.choice(["North", "South", "East", "West"], size=n_rows, p=[0.3, 0.25, 0.25, 0.2]),
    "Product_Category": np.random.choice(["Electronics", "Clothing", "Home & Kitchen", "Books"], size=n_rows),
    "Units_Sold": np.random.randint(1, 15, size=n_rows),
    "Unit_Price": np.random.choice([15.0, 25.5, 49.99, 120.0, 299.99], size=n_rows),
    "Customer_Rating": np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0, np.nan], size=n_rows, p=[0.05, 0.1, 0.2, 0.35, 0.25, 0.05])
}

# Convert dictionary to Pandas DataFrame
df_raw = pd.DataFrame(raw_data)

# Inject some missing values into 'Units_Sold' to simulate real messy data
df_raw.loc[df_raw.sample(15).index, "Units_Sold"] = np.nan

# Save as a raw CSV file
df_raw.to_csv("raw_sales_data.csv", index=False)

print("Raw dataset created successfully and saved as 'raw_sales_data.csv'!")

Raw dataset created successfully and saved as 'raw_sales_data.csv'!


In [3]:
# 1. Load the raw CSV file into a Pandas DataFrame
df = pd.read_csv("raw_sales_data.csv")

# 2. Check initial missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())
print("\n" + "=" * 40 + "\n")

# 3. Clean Missing Data (Imputation)
# Fill missing Units_Sold with median
df["Units_Sold"] = df["Units_Sold"].fillna(df["Units_Sold"].median())

# Fill missing Customer_Rating with average (rounded to 1 decimal place)
df["Customer_Rating"] = df["Customer_Rating"].fillna(
    round(df["Customer_Rating"].mean(), 1)
)

# 4. Feature Engineering: Compute Total Revenue for each order
df["Total_Revenue"] = df["Units_Sold"] * df["Unit_Price"]

# 5. Format Date column
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.strftime("%Y-%m")

print("--- Missing Values After Cleaning ---")
print(df.isnull().sum())
print("\n" + "=" * 40 + "\n")

# Display the first 5 rows of our cleaned data
print("--- Cleaned Sample Data ---")
df.head()

--- Missing Values Before Cleaning ---
Order_ID             0
Date                 0
Region               0
Product_Category     0
Units_Sold          15
Unit_Price           0
Customer_Rating     24
dtype: int64


--- Missing Values After Cleaning ---
Order_ID            0
Date                0
Region              0
Product_Category    0
Units_Sold          0
Unit_Price          0
Customer_Rating     0
Total_Revenue       0
Month               0
dtype: int64


--- Cleaned Sample Data ---


,Order_ID,Date,Region,Product_Category,Units_Sold,Unit_Price,Customer_Rating,Total_Revenue,Month
0,ORD-1000,2025-04-13,North,Books,8.0,299.99,4.0,2399.92,2025-04
1,ORD-1001,2025-06-29,North,Electronics,3.0,120.00,4.0,360.00,2025-06
2,ORD-1002,2025-04-03,North,Home & Kitchen,11.0,299.99,1.0,3299.89,2025-04
3,ORD-1003,2025-01-15,South,Home & Kitchen,12.0,15.00,2.0,180.00,2025-01
4,ORD-1004,2025-04-17,North,Home & Kitchen,13.0,15.00,5.0,195.00,2025-04


In [4]:
# 1. Category-wise Summary
category_summary = (
    df.groupby("Product_Category")
    .agg(
        Total_Orders=("Order_ID", "count"),
        Total_Units=("Units_Sold", "sum"),
        Total_Revenue=("Total_Revenue", "sum"),
        Avg_Rating=("Customer_Rating", "mean"),
    )
    .reset_index()
    .round(2)
)

# 2. Region-wise Summary
region_summary = (
    df.groupby("Region")
    .agg(
        Total_Orders=("Order_ID", "count"),
        Total_Revenue=("Total_Revenue", "sum"),
    )
    .reset_index()
    .sort_values(by="Total_Revenue", ascending=False)
    .round(2)
)

print("--- Category Performance Summary ---")
print(category_summary)
print("\n" + "=" * 40 + "\n")

print("--- Regional Sales Performance ---")
print(region_summary)

--- Category Performance Summary ---
  Product_Category  Total_Orders  Total_Units  Total_Revenue  Avg_Rating
0            Books           106        842.0       85337.66        3.79
1         Clothing           129        999.0      105755.87        3.70
2      Electronics           135       1031.0      119928.62        3.64
3   Home & Kitchen           130       1003.0      111652.51        3.73


--- Regional Sales Performance ---
  Region  Total_Orders  Total_Revenue
1  North           137      135222.38
2  South           141      117916.65
0   East           134      101501.61
3   West            88       68034.02


In [5]:
# Define output filename
output_filename = "Automated_Sales_Report.xlsx"

# Use ExcelWriter with openpyxl engine to write to multiple sheets
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:

    # Sheet 1: Executive Summary (Category & Regional Performance)
    category_summary.to_excel(
        writer, sheet_name="Executive Summary", startrow=1, index=False
    )
    region_summary.to_excel(
        writer, sheet_name="Executive Summary", startrow=10, index=False
    )

    # Sheet 2: Cleaned Transactional Data
    df.to_excel(writer, sheet_name="Cleaned Data", index=False)

print(
    f"Success! Report generated and saved as '{output_filename}' in your files."
)

Success! Report generated and saved as 'Automated_Sales_Report.xlsx' in your files.
